In [3]:
import torch

x = torch.ones(5)
x

tensor([1., 1., 1., 1., 1.])

In [4]:
y = torch.zeros(3)
y

tensor([0., 0., 0.])

In [6]:
w = torch.rand(5, 3, requires_grad=True)
print(w)

tensor([[0.2948, 0.7198, 0.9224],
        [0.1925, 0.5444, 0.3714],
        [0.7028, 0.1421, 0.3841],
        [0.3536, 0.8121, 0.1900],
        [0.0254, 0.0353, 0.0432]], requires_grad=True)


In [7]:
b = torch.rand(3, requires_grad=True)
print(b)

tensor([0.3369, 0.2305, 0.1868], requires_grad=True)


In [8]:
z = torch.matmul(x, w) + b
print(z)

tensor([1.9061, 2.4841, 2.0979], grad_fn=<AddBackward0>)


In [9]:

loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x17d167430>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x17d167430>


In [11]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.2902, 0.3077, 0.2969],
        [0.2902, 0.3077, 0.2969],
        [0.2902, 0.3077, 0.2969],
        [0.2902, 0.3077, 0.2969],
        [0.2902, 0.3077, 0.2969]])
tensor([0.2902, 0.3077, 0.2969])


In [10]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

True
False


In [12]:
z = torch.matmul(x, w) + b
z_det = z.detach()
print(z_det.requires_grad)

False


In [15]:
# 创建一个 4 * 5 的单位举证
inp = torch.eye(4, 5, requires_grad=True)
print(inp)

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.]], requires_grad=True)


In [16]:
# inp + 1 每个元素 +1; pow(2) 每个元素平方；t() 举证转置
# out = ((inp + 1)²)ᵀ 数学表达式
out = (inp+1).pow(2).t()
print(out)

tensor([[4., 1., 1., 1.],
        [1., 4., 1., 1.],
        [1., 1., 4., 1.],
        [1., 1., 1., 4.],
        [1., 1., 1., 1.]], grad_fn=<TBackward0>)


In [17]:
# 反向传播计算梯度：
# torch.ones_like(out) 创建一个与 out 形状相同的 全 1 张量做为梯度
# retain_graph=True：保留计算图，允许再次反向传播
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


In [18]:
# 第二次反向传播，梯度会累积
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")


Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])


In [19]:
# inp.grad.zero_() 将梯度清零
inp.grad.zero_()
# 第三次反向传播
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")


Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


### 数学推导

对于每个操作，我们可以计算具体的梯度：

inp + 1 的梯度：∂(x+1)/∂x = 1

.pow(2) 的梯度：∂(x²)/∂x = 2x

.t() 的梯度：转置操作的梯度也是转置

所以对于 out = ((inp + 1)²)ᵀ，梯度为： ∂out/∂inp = 2 × (inp + 1)

由于 inp 是单位矩阵，所以：

inp + 1 =

[

    [2, 1, 1, 1, 1],

    [1, 2, 1, 1, 1],

    [1, 1, 2, 1, 1],

    [1, 1, 1, 2, 1]
]

梯度 = 2 × (inp + 1) =
[

    [4, 2, 2, 2, 2],

    [2, 4, 2, 2, 2],

    [2, 2, 4, 2, 2],

    [2, 2, 2, 4, 2]
]

### 关键知识点

1. 梯度累积：PyTorch 默认会累积梯度，多次调用 .backward() 会导致梯度相加
2. 梯度清零：训练神经网络时，通常需要在每个 batch 前调用 .zero_grad() 清除之前的梯度
3. retain_graph：允许重复使用计算图进行多次反向传播
4. 梯度传递：.backward() 中的参数指定了反向传播的初始梯度